# rf yield model v1.4

diffs from v1.1:
1. **detrended tgt**: lin trend per state on `yield ~ year`, rf on resids, trend added back at pred
2. **ext train win**: train 05-23, val 24
3. **bias corr**: state mean val err sub'd from final preds

out: `outputs/predictions.05_model1.4.csv`

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from pathlib import Path

NOTEBOOK_NAME = '05_model1.4'
OUTPUT_CSV    = f"../outputs/predictions.{NOTEBOOK_NAME}.csv"
print(f"Output will be saved to: {OUTPUT_CSV}")

df = pd.read_csv("../data/processed/training_features.csv")
print(f"Loaded {len(df)} rows, {df.shape[1]} columns")
print(df.columns.tolist())

In [ ]:
# feat sets (same as v1.1, w/ yr)
FEATURE_SETS = {
    'aug1':  ['year','tavg_may','prcp_may','tavg_jun','prcp_jun','tavg_jul','prcp_jul',
              'ndvi_aug1'],
    'sep1':  ['year','tavg_may','prcp_may','tavg_jun','prcp_jun','tavg_jul','prcp_jul',
              'tavg_aug','prcp_aug','ndvi_aug1','ndvi_sep1'],
    'oct1':  ['year','tavg_may','prcp_may','tavg_jun','prcp_jun','tavg_jul','prcp_jul',
              'tavg_aug','prcp_aug','tavg_sep','prcp_sep',
              'ndvi_aug1','ndvi_sep1','ndvi_oct1'],
    'final': ['year','tavg_may','prcp_may','tavg_jun','prcp_jun','tavg_jul','prcp_jul',
              'tavg_aug','prcp_aug','tavg_sep','prcp_sep','tavg_oct','prcp_oct',
              'ndvi_aug1','ndvi_sep1','ndvi_oct1','ndvi_final'],
}
TARGET = 'yield_bu_acre'

In [ ]:
# detrend tgt per state

trend_models = {}  # state -> fitted LinearRegression

historical = df[df['year'] <= 2024].copy()

for state in historical['state'].unique():
    s = historical[historical['state'] == state][['year', TARGET]].dropna()
    lr = LinearRegression()
    lr.fit(s[['year']], s[TARGET])
    trend_models[state] = lr
    slope = lr.coef_[0]
    print(f"{state:12s}  trend: +{slope:.2f} bu/acre/year  "
          f"(2025 trend value: {lr.predict([[2025]])[0]:.1f})")

# add detrended col
def detrend_yield(row):
    if pd.isna(row[TARGET]):
        return np.nan
    trend_val = trend_models[row['state']].predict([[row['year']]])[0]
    return row[TARGET] - trend_val

df['yield_detrended'] = df.apply(detrend_yield, axis=1)
TARGET_DETRENDED = 'yield_detrended'

print("\nDetrend check (residuals should be centred near 0 per state):")
print(df.groupby('state')['yield_detrended'].agg(['mean','std']).round(2))

In [ ]:
# ext train: 05-23, val 24
df_enc = pd.get_dummies(df, columns=['state'])
state_dummies = [c for c in df_enc.columns if c.startswith('state_')]

train = df_enc[df_enc['year'] <= 2023].copy()
test  = df_enc[df_enc['year'] == 2024].copy()
pred  = df_enc[df_enc['year'] == 2025].copy()

if len(pred) == 0:
    raise ValueError(
        "No 2025 rows in training_features.csv. "
        "Ensure 02_weather.ipynb fetched 2025 data, then re-run 04_merge_features.ipynb."
    )

print(f"Train: {len(train)} | Validate: {len(test)} | Predict: {len(pred)}")
print(f"Train years: {sorted(df_enc[df_enc['year'] <= 2023]['year'].unique())}")
print(f"Val year:    {sorted(test['year'].unique())}")

In [ ]:
# get orig state from 1-hot row
def get_state_name(row):
    matched = [c.replace('state_', '') for c in state_dummies if row.get(c, 0) == 1]
    return matched[0] if matched else 'UNKNOWN'

pred_states = [get_state_name(row) for _, row in pred.iterrows()]
test_states = [get_state_name(row) for _, row in test.iterrows()]

In [ ]:
# main loop
results      = []
bias_by_state = {}  # acc state bias across scored dates

for date_label, base_features in FEATURE_SETS.items():
    features = [f for f in base_features + state_dummies if f in df_enc.columns]

    col_means = train[features].mean()
    X_train   = train[features].fillna(col_means)
    X_test    = test[features].fillna(col_means)
    X_pred    = pred[features].fillna(col_means)

    # train on detrended resids
    y_train_detrended = train[TARGET_DETRENDED]
    y_test_raw        = test[TARGET]   # orig yield for rmse

    model = RandomForestRegressor(
        n_estimators=200, max_depth=10, min_samples_leaf=2,
        random_state=42, n_jobs=-1
    )
    model.fit(X_train, y_train_detrended)

    # add trend back at pred time
    def retrend(residuals, states, year):
        """add state trend val for yr to rf resids"""
        return np.array([
            res + trend_models[st].predict([[year]])[0]
            for res, st in zip(residuals, states)
        ])

    test_resid  = model.predict(X_test)
    test_preds  = retrend(test_resid, test_states, 2024)
    val_rmse    = np.sqrt(mean_squared_error(y_test_raw, test_preds))

    # calc state bias on val set
    # bias = mean(pred - act) per state on val yr
    # acc across dates, avg at end
    for st, pred_val, actual_val in zip(test_states, test_preds, y_test_raw):
        bias_by_state.setdefault(st, []).append(pred_val - actual_val)

    print(f"{date_label} — Validation RMSE (pre-correction): {val_rmse:.2f} bu/acre")

    # pred 25 resids, retrend
    pred_resid  = model.predict(X_pred)
    point_preds = retrend(pred_resid, pred_states, 2025)

    # boot uncert (detrended resids, retrend each boot)
    boot_preds = []
    for _ in range(500):
        idx = np.random.choice(len(X_train), len(X_train), replace=True)
        m = RandomForestRegressor(n_estimators=30, max_depth=10,
                                  random_state=None, n_jobs=-1)
        m.fit(X_train.iloc[idx], y_train_detrended.iloc[idx])
        boot_resid = m.predict(X_pred)
        boot_preds.append(retrend(boot_resid, pred_states, 2025))

    boot_arr = np.array(boot_preds)
    ci_lower = np.percentile(boot_arr, 5,  axis=0)
    ci_upper = np.percentile(boot_arr, 95, axis=0)

    # analog yrs (detrended feat space)
    scaler         = StandardScaler().fit(X_train)
    X_train_sc     = scaler.transform(X_train)
    X_pred_sc      = scaler.transform(X_pred)
    analog_years_list = []
    for pred_vec in X_pred_sc:
        dists    = np.linalg.norm(X_train_sc - pred_vec, axis=1)
        top3_idx = np.argsort(dists)[:3]
        analog_years_list.append(train.iloc[top3_idx]['year'].tolist())

    for i, state in enumerate(pred_states):
        results.append({
            'state':                state,
            'forecast_date':        date_label,
            'predicted_yield_raw':  round(point_preds[i], 2),   # pre-bias corr
            'ci_lower_raw':         round(ci_lower[i], 2),
            'ci_upper_raw':         round(ci_upper[i], 2),
            'analog_years':         str(analog_years_list[i]),
            'val_rmse_pre':         round(val_rmse, 2),
        })

In [ ]:
# ply state bias corr
# mean bias = mean(pred - act) across scored dates on val yr
# sub from preds & shift ci

mean_bias = {st: np.mean(errs) for st, errs in bias_by_state.items()}
print("Per-state mean bias (validation year, pre-correction):")
for st, b in sorted(mean_bias.items()):
    direction = 'over' if b > 0 else 'under'
    print(f"  {st:12s}  {b:+.1f} bu/acre ({direction}-predicted)")

results_df = pd.DataFrame(results)

results_df['bias_correction'] = results_df['state'].map(mean_bias).round(2)
results_df['predicted_yield'] = (results_df['predicted_yield_raw'] - results_df['bias_correction']).round(2)
results_df['ci_lower']        = (results_df['ci_lower_raw']        - results_df['bias_correction']).round(2)
results_df['ci_upper']        = (results_df['ci_upper_raw']        - results_df['bias_correction']).round(2)

# recalc val_rmse post-corr (approx)
# rpt corr ver in summ
print("\nBias correction applied.")

In [ ]:
# save & disp
out_cols = [
    'state', 'forecast_date', 'predicted_yield',
    'ci_lower', 'ci_upper', 'analog_years',
    'val_rmse_pre', 'bias_correction', 'predicted_yield_raw'
]
results_df[out_cols].to_csv(OUTPUT_CSV, index=False)

print(f"Saved {len(results_df)} rows to {OUTPUT_CSV}")
print()
print(results_df[out_cols].sort_values(['forecast_date','state']).to_string(index=False))